# 02 · Risk Analytics
**VaR · CVaR · Sharpe · Sortino · Beta · Drawdown**

This notebook runs the full risk engine on your live portfolio:
- **VaR/CVaR** at 95% and 99% confidence (1-day and 10-day)
- **Sharpe / Sortino / Calmar** ratios
- **Portfolio Beta** vs SPY
- **Maximum Drawdown** and duration
- **Rolling volatility and beta** (regime detection)
- **Correlation heatmap** (diversification check)

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})
print("Libraries loaded ✓")

## 1 · Load Data

In [ ]:
from src.data_fetcher import build_portfolio_snapshot, get_historical_prices
from src.portfolio    import compute_portfolio_returns, get_benchmark_returns, compute_weights

snap    = build_portfolio_snapshot(use_sheets=True)
snap    = compute_weights(snap)
tickers = snap["ticker"].tolist()

print("Fetching 2-year price history …")
prices  = get_historical_prices(tickers, period="2y")
port_ret = compute_portfolio_returns(prices, snap)
bench_ret = get_benchmark_returns("SPY", period="2y")

# Align
common = port_ret.index.intersection(bench_ret.index)
port_ret  = port_ret.loc[common]
bench_ret = bench_ret.loc[common]

print(f"Date range: {common[0].date()} → {common[-1].date()} ({len(common)} days)")

## 2 · Full Risk Report

In [ ]:
from src.portfolio import portfolio_summary
from src.risk import full_risk_report

s   = portfolio_summary(snap)
rpt = full_risk_report(port_ret, bench_ret, s["total_value_krw"])

var = rpt["var"]
print(f"""
╔══════════════════════════════════════════════════════════════╗
  Risk Report  (95% confidence unless noted)
╠══════════════════════════════════════════════════════════════╣
  Annualised Return  :  {rpt['annualised_return_pct']:+.2f}%
  Annualised Vol     :  {rpt['annualised_vol_pct']:.2f}%
  Sharpe Ratio       :  {rpt['sharpe_ratio']:.3f}
  Sortino Ratio      :  {rpt['sortino_ratio']:.3f}
  Calmar Ratio       :  {rpt['calmar_ratio']:.3f}
╠══════════════════════════════════════════════════════════════╣
  Maximum Drawdown   :  -{rpt['max_drawdown_pct']:.2f}%
  Max DD Duration    :  {rpt['drawdown_duration']['max_duration_days']} days
  Currently in DD    :  {'Yes ⚠️' if rpt['drawdown_duration']['in_drawdown'] else 'No ✅'}
╠══════════════════════════════════════════════════════════════╣
  Portfolio Beta     :  {rpt['beta']:.3f}  (vs SPY)
  Tracking Error     :  {rpt['tracking_error_pct']:.2f}%
  Information Ratio  :  {rpt['information_ratio']:.3f}
╠══════════════════════════════════════════════════════════════╣
  1-Day VaR  (hist)  :  -{var['hist_var_1d_pct']:.3f}%  /  ₩{var['hist_var_1d_krw']:,.0f}
  1-Day CVaR (hist)  :  -{var['hist_cvar_1d_pct']:.3f}%  /  ₩{var['hist_cvar_1d_krw']:,.0f}
  1-Day VaR  (param) :  -{var['param_var_1d_pct']:.3f}%
  10-Day VaR (hist)  :  -{var['hist_var_10d_pct']:.3f}%  /  ₩{var['hist_var_10d_krw']:,.0f}
╚══════════════════════════════════════════════════════════════╝""")

## 3 · Cumulative Return vs SPY

In [ ]:
from src.portfolio import compute_cumulative_returns

cum_port  = compute_cumulative_returns(port_ret)
cum_bench = compute_cumulative_returns(bench_ret)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_port.index,  (cum_port  - 1) * 100, label="My Portfolio", lw=2, color="#4C72B0")
ax.plot(cum_bench.index, (cum_bench - 1) * 100, label="SPY",          lw=2, color="#DD8452", ls="--")
ax.set_ylabel("Cumulative Return (%)")
ax.set_title("Portfolio vs SPY — 2-Year Cumulative Return", fontsize=13)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
plt.tight_layout()
plt.show()

## 4 · Drawdown Chart

In [ ]:
from src.risk import drawdown_series

dd = drawdown_series(port_ret) * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(dd.index, dd, 0, alpha=0.5, color="#e74c3c", label="Drawdown")
ax.plot(dd.index, dd, color="#c0392b", lw=1)
ax.set_ylabel("Drawdown (%)")
ax.set_title(f"Portfolio Drawdown  (Max: {dd.min():.2f}%)", fontsize=13)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
plt.tight_layout()
plt.show()

## 5 · Rolling Volatility & Beta

In [ ]:
from src.risk import rolling_volatility, rolling_beta

rv  = rolling_volatility(port_ret, window=21) * 100
rb  = rolling_beta(port_ret, bench_ret, window=63)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(rv.index, rv, color="#4C72B0", lw=1.5)
ax1.set_ylabel("Annualised Vol (%)")
ax1.set_title("21-Day Rolling Volatility", fontsize=12)
ax1.axhline(rv.mean(), color="orange", ls="--", lw=1, label=f"Avg {rv.mean():.1f}%")
ax1.legend()

ax2.plot(rb.index, rb, color="#2ca02c", lw=1.5)
ax2.axhline(1.0, color="grey",   ls="--", lw=1, label="β=1 (market)")
ax2.axhline(rb.mean(), color="orange", ls="--", lw=1, label=f"Avg β={rb.mean():.2f}")
ax2.set_ylabel("Beta vs SPY")
ax2.set_title("63-Day Rolling Beta", fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

## 6 · Correlation Heatmap

In [ ]:
from src.risk import correlation_matrix

corr = correlation_matrix(prices)

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    annot_kws={"size": 8}, ax=ax
)
ax.set_title("Return Correlation Matrix (2Y Daily)", fontsize=13)
plt.tight_layout()
plt.show()

# Flag highly correlated pairs (>0.80)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i,j]) > 0.80:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i,j],3)))

if high_corr:
    print("⚠️  Highly correlated pairs (>0.80) — low diversification benefit:")
    for a, b, c in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"   {a:10s} ↔ {b:10s}  ρ = {c:+.3f}")
else:
    print("✅ No pairs with correlation > 0.80")

## 7 · VaR Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(port_ret * 100, bins=80, color="#4C72B0", edgecolor="white", alpha=0.8)
ax.axvline(-rpt["var"]["hist_var_1d_pct"],  color="#e74c3c", lw=2, ls="--",
           label=f"95% VaR  −{rpt['var']['hist_var_1d_pct']:.2f}%")
ax.axvline(-rpt["var"]["hist_cvar_1d_pct"], color="#c0392b", lw=2, ls=":",
           label=f"95% CVaR −{rpt['var']['hist_cvar_1d_pct']:.2f}%")
ax.set_xlabel("Daily Return (%)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Daily Portfolio Returns", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()